# Retail Sales and Customer Analysis

What drives this store’s revenue, and how important are repeat customers?

First session: load, understand, and audit the source data. One row is an invoice line; count distinct invoices to count orders.

In [9]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_FILE = ROOT / "data" / "raw" / "Online Retail.xlsx"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Download Online Retail.xlsx from UCI and place it at {DATA_FILE}")
df = pd.read_excel(DATA_FILE)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Understand the columns

| Column | Meaning |
|---|---|
| InvoiceNo | Invoice identifier; invoices contain multiple lines |
| StockCode | Product identifier |
| Description | Product description |
| Quantity | Units on this invoice line |
| InvoiceDate | Transaction date and time |
| UnitPrice | Price per unit |
| CustomerID | Customer identifier |
| Country | Customer country |

In [8]:
print("Shape:", df.shape)
df.info()
display(df.isna().sum().rename("missing_rows").to_frame())
display(df.describe(include="all"))

Shape: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  string        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  string        
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), object(2), str(1), string(2)
memory usage: 33.1+ MB


,missing_rows
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,406829,541909
unique,25900,4070,4223,NaN,NaN,NaN,4372,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,17841.0,United Kingdom
freq,1114,2313,2369,NaN,NaN,NaN,7983,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386,4.611114,NaN,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,NaN,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,NaN,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,NaN,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,NaN,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,NaN,NaN


## Audit before cleaning

Counts can overlap. Negative quantities may represent returns or adjustments. Do not remove unusual rows until their meaning is understood.

In [7]:
duplicate_count = int(df.duplicated().sum())
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["CustomerID"] = df["CustomerID"].astype("string")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="raise")
is_cancellation = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
issues = {
    "Rows with any missing value": int(df.isna().any(axis=1).sum()),
    "Missing customer ID": int(df["CustomerID"].isna().sum()),
    "Cancellation rows": int(is_cancellation.sum()),
    "Nonpositive quantities": int((df["Quantity"] <= 0).sum()),
    "Nonpositive prices": int((df["UnitPrice"] <= 0).sum()),
    "Exact duplicate rows": duplicate_count,
}
audit = pd.DataFrame({"issue": issues.keys(), "affected_rows": issues.values()})
audit["decision"] = "Pending review; source data preserved"
display(audit)
print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

,issue,affected_rows,decision
0,Rows with any missing value,135080,Pending review; source data preserved
1,Missing customer ID,135080,Pending review; source data preserved
2,Cancellation rows,9288,Pending review; source data preserved
3,Nonpositive quantities,10624,Pending review; source data preserved
4,Nonpositive prices,2517,Pending review; source data preserved
5,Exact duplicate rows,5268,Pending review; source data preserved


Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [6]:
month_coverage = df.groupby(df["InvoiceDate"].dt.to_period("M")).agg(
    first_observed=("InvoiceDate", "min"),
    last_observed=("InvoiceDate", "max"),
    invoice_lines=("InvoiceNo", "size"),
)
month_coverage["review_note"] = "Review calendar coverage; transaction dates alone do not prove completeness"
display(month_coverage)

,first_observed,last_observed,invoice_lines,review_note
InvoiceDate,,,,
2010-12,2010-12-01 08:26:00,2010-12-23 17:41:00,42481,Review calendar coverage; transaction dates al...
2011-01,2011-01-04 10:00:00,2011-01-31 16:59:00,35147,Review calendar coverage; transaction dates al...
2011-02,2011-02-01 08:23:00,2011-02-28 17:04:00,27707,Review calendar coverage; transaction dates al...
2011-03,2011-03-01 08:30:00,2011-03-31 19:55:00,36748,Review calendar coverage; transaction dates al...
2011-04,2011-04-01 08:22:00,2011-04-28 19:53:00,29916,Review calendar coverage; transaction dates al...
2011-05,2011-05-01 10:51:00,2011-05-31 15:53:00,37030,Review calendar coverage; transaction dates al...
2011-06,2011-06-01 07:37:00,2011-06-30 20:08:00,36874,Review calendar coverage; transaction dates al...
2011-07,2011-07-01 08:16:00,2011-07-31 16:04:00,39518,Review calendar coverage; transaction dates al...
2011-08,2011-08-01 08:30:00,2011-08-31 17:45:00,35284,Review calendar coverage; transaction dates al...


## Initial audit results and decisions

The source contains **541,909 invoice lines** and eight columns, dated **1 December 2010 to 9 December 2011**. Invoice lines are not orders.

| Issue | Affected rows | Decision for the next analysis stage |
|---|---:|---|
| Missing customer ID | 135,080 | Preserve for overall purchase sales; exclude from customer metrics |
| Missing description | 1,454 | Preserve; use StockCode for product grouping and inspect labels |
| Cancellation invoice | 9,288 | Preserve in raw data; exclude from purchase sales and review separately |
| Quantity at or below zero | 10,624 | Preserve in raw data; exclude from positive purchase sales; investigate returns and adjustments |
| Unit price at or below zero | 2,517 | Preserve in raw data; exclude from purchase sales; inspect free items and adjustments |
| Exact duplicate rows | 5,268 | Flag and retain pending evidence; identical lines may be legitimate, so do not automatically drop |

Counts overlap. No rows have been removed. **December 2011 is a partial month** and should be labelled or excluded from full-month comparisons. A lack of transactions on individual days does not prove that a month is incomplete; December 2010 ends on 23 December and requires calendar context before interpreting it.

Next: build the positive purchase-sales dataset, load SQLite, and validate totals before drawing business conclusions. Purchase revenue will not subtract returns and cancellations, so it must not be labelled net revenue.


## Create purchase sales and SQLite database

Keep positive quantities and prices, excluding cancellation invoices. Missing customer IDs remain in overall sales and are excluded from customer analysis. Duplicate lines remain pending review. Purchase revenue does not subtract returns and is not net revenue. Run the earlier cells first.

In [4]:
# Keep positive purchases and exclude cancellation invoices.
sales = df.loc[
    ~is_cancellation
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
].copy()

sales["LineRevenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["InvoiceMonth"] = sales["InvoiceDate"].dt.to_period("M").astype(str)

# Missing customer IDs stay in overall sales.
customer_sales = sales.dropna(subset=["CustomerID"]).copy()

processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)
sales.to_csv(processed / "sales.csv", index=False)

# Rerunning replaces the generated sales table with the current data.
with sqlite3.connect(processed / "retail.db") as conn:
    sales.to_sql("sales", conn, if_exists="replace", index=False)

print(f"Saved {len(sales):,} purchase lines.")

Saved 530,104 purchase lines.


## First SQL summary

Orders count distinct invoices, not invoice lines. Average order value divides purchase revenue by the number of orders.

In [5]:
with sqlite3.connect(processed / "retail.db") as conn:
    summary = pd.read_sql_query("""
        SELECT
            COUNT(DISTINCT InvoiceNo) AS orders,
            SUM(LineRevenue) AS purchase_revenue,
            SUM(LineRevenue) / NULLIF(COUNT(DISTINCT InvoiceNo), 0)
                AS average_order_value
        FROM sales
    """, conn)

display(summary)

,orders,purchase_revenue,average_order_value
0,19960,1.066668e+07,534.403033
